# Checking the binary mask saved as nii.gz

In [10]:
path_niigz = "C:\\Users\\kaiak\\OneDrive\\Documents\\UCLA\\SAVAGELAB\\AngicartC++\\AngicartCpp\\sample_outputs\\FITC-MCA0_N12_PI001_s1_vessels.nii.gz"

In [11]:
import nibabel as nib
import numpy as np
import plotly.graph_objects as go
from skimage.measure import marching_cubes

def visualize_nii_surface(file_path, threshold=0.5, step_size=2):
    """
    Loads a .nii.gz binary volume and displays an interactive 3D surface mesh.
    
    Parameters:
        file_path (str): Path to the .nii.gz file.
        threshold (float): Surface extraction threshold (0.5 works well for 0/1 binary masks).
        step_size (int): Downsampling step for Marching Cubes (increase for faster rendering).
    """
    # 1. Load the NIfTI volume
    img = nib.load(file_path)
    volume = img.get_fdata()
    
    # Get voxel spacing (zooms) to ensure correct physical 3D proportions (mm)
    spacing = img.header.get_zooms()[:3]
    
    # Quick safety check for empty volume
    if not np.any(volume >= threshold):
        print("Error: The volume contains no binary mask above the threshold.")
        return

    # 2. Extract 3D Surface Mesh via Marching Cubes
    # 'spacing' scales the vertices so the 3D model maintains accurate physical dimensions
    verts, faces, normals, values = marching_cubes(
        volume, 
        level=threshold, 
        spacing=spacing, 
        step_size=step_size
    )

    # 3. Build Interactive 3D Mesh in Plotly
    fig = go.Figure(data=[
        go.Mesh3d(
            x=verts[:, 0],
            y=verts[:, 1],
            z=verts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            color='cyan',
            opacity=0.8,
            lighting=dict(
                ambient=0.4,
                diffuse=0.8,
                fresnel=0.2,
                specular=0.5,
                roughness=0.5
            ),
            lightposition=dict(x=100, y=200, z=150)
        )
    ])

    # 4. Set Layout Options
    fig.update_layout(
        title=f"3D Surface Rendering: {file_path}",
        scene=dict(
            xaxis_title="X (mm)",
            yaxis_title="Y (mm)",
            zaxis_title="Z (mm)",
            aspectmode='data'  # Preserves 1:1:1 aspect ratio based on real coordinates
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    # Open interactively in default browser or Jupyter environment
    fig.show()

In [12]:
visualize_nii_surface(path_niigz)